In [ ]:
import os
import polars as pl

if not os.path.exists("datasets"):
    os.mkdir("datasets")

In [ ]:
# Buscamos el nombre del archivo parquet del libro

book_file = [f for f in os.listdir("datasets") if "dataset_completo" in f and f.endswith(".parquet")][0]
path_book = os.path.join("datasets", book_file)

# Cargamos una vista previa
dataset_completo= pl.scan_parquet(path_book)

In [ ]:
import polars as pl
import polars.selectors as cs

# 1. X_features: Todas las columnas MENOS las que empiezan por "label"
X_features = dataset_completo.select(~cs.starts_with("label"))

# 2. y_labels: SOLO las columnas que empiezan por "label"
y_labels = dataset_completo.select(cs.starts_with("label"))


In [ ]:
def finalize_dataset(
    df_features: pl.DataFrame,
    df_labels: pl.DataFrame,
) -> pl.DataFrame:
    """Une features y labels y elimina los nulos generados por ventanas móviles.

    No aplicamos Z-score aquí: la normalización se hará más adelante,
    ajustando el StandardScaler SOLO con datos de entrenamiento de cada
    fold (y, para el modelo final, con todo el pool de CV), evitando así
    fuga de información (data leakage) hacia validación/test.
    """
    full_df = pl.concat([df_features, df_labels], how="horizontal")
    cleaned_df = full_df.drop_nulls()
    return cleaned_df


# Ensamblar y limpiar el dataset final
dataset_final = finalize_dataset(X_features, y_labels)

# Eliminamos columnas que no son features numéricas de entrada al modelo
dataset_final = dataset_final.drop([
    pl.col("mid_price_coordenada"),
    pl.col("^feat_.*$"),  # expresiones regulares nativas de Polars
])



In [ ]:
import polars as pl
import pandas as pd
import numpy as np

TARGET_COL = "label_k_100"
CORR_THRESHOLD = 0.90  # Umbral para eliminar variables redundantes entre sí

# 1. Separar columnas
all_columns = dataset_final.collect_schema().names()
label_cols = [c for c in all_columns if c.startswith("label")]
candidate_feature_cols = [c for c in all_columns if c not in label_cols]

print(f"Calculando matriz de correlación para {len(candidate_feature_cols)} características...")

# 2. Construir expresiones para la matriz de correlación cruzada (todas contra todas)
# Nota: Si tienes miles de columnas, considera aplicar un .head(50000).collect() previo.
matrix_exprs = []
for c1 in candidate_feature_cols:
    for c2 in candidate_feature_cols:
        if c1 >= c2:  # Solo calculamos la mitad para ahorrar tiempo y procesamiento
            matrix_exprs.append(pl.corr(pl.col(c1), pl.col(c2)).alias(f"{c1}__VS__{c2}"))

# Calcular los valores en Polars
corr_results = dataset_final.select(matrix_exprs).collect()

# 3. Reconstruir la matriz simétrica en Pandas para usar sus herramientas de filtrado rápido
corr_matrix_pd = pd.DataFrame(index=candidate_feature_cols, columns=candidate_feature_cols, dtype=float)

for col in corr_results.columns:
    c1, c2 = col.split("__VS__")
    val = corr_results.get_column(col)[0]
    corr_matrix_pd.loc[c1, c2] = val
    corr_matrix_pd.loc[c2, c1] = val

# Llenar la diagonal con 1 por seguridad
np.fill_diagonal(corr_matrix_pd.values, 1.0)
corr_matrix_abs = corr_matrix_pd.abs()

# 4. Encontrar las características redundantes (con correlación > umbral entre ellas)
# Usamos un triángulo superior para no evaluar el mismo par dos veces
upper_tri = corr_matrix_abs.where(np.triu(np.ones(corr_matrix_abs.shape), k=1).astype(bool))

# Identificar qué columnas descartar
cols_to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > CORR_THRESHOLD)]

print(f"\n🔻 Columnas eliminadas por alta correlación ENTRE SÍ (> {CORR_THRESHOLD}) ({len(cols_to_drop)}):")
print(cols_to_drop)

# 5. Crear el nuevo dataset final sin las características redundantes
# Conservamos todas las labels intactas y las features que pasaron el filtro
final_features_to_keep = [c for c in candidate_feature_cols if c not in cols_to_drop]
dataset_filtrado = dataset_final.select(final_features_to_keep + label_cols)


Calculando matriz de correlación para 144 características...

🔻 Columnas eliminadas por alta correlación ENTRE SÍ (> 0.9) (74):
['v3', 'v5', 'v7', 'v9', 'v11', 'v13', 'v15', 'v17', 'v19', 'v21', 'v23', 'v25', 'v27', 'v29', 'v31', 'v33', 'v35', 'v37', 'v39', 'v47', 'v48', 'v49', 'v50', 'v57', 'v58', 'v59', 'v60', 'v67', 'v68', 'v69', 'v70', 'v72', 'v73', 'v74', 'v75', 'v76', 'v77', 'v78', 'v79', 'v80', 'v82', 'v83', 'v84', 'v85', 'v86', 'v87', 'v88', 'v89', 'v90', 'v92', 'v93', 'v94', 'v95', 'v96', 'v97', 'v98', 'v99', 'v100', 'v101', 'v102', 'v103', 'v104', 'v105', 'v106', 'v107', 'v108', 'v109', 'v110', 'v132', 'v137', 'v139', 'v140', 'v142', 'v143']


In [ ]:
import polars as pl


def finalize_dataset_for_xgboost(
    df_features: pl.DataFrame,
    df_labels: pl.DataFrame,
    apply_zscore: bool = False,
) -> pl.DataFrame:
    """Une las características (features) y las etiquetas (labels), elimina

    los valores nulos producidos por las ventanas móviles, y opcionalmente
    aplica normalización Z-score.
    """
    # 1. CONCATENACIÓN HORIZONTAL
    # Juntamos las 144 features (+ el OBI) con las columnas de etiquetas calculadas
    full_df = pl.concat([df_features, df_labels], how="horizontal")

    # 2. LIMPIEZA DE NULOS CRÍTICA
    # Al inicio del dataset, las ventanas 'rolling' (de tamaño 10, 50, 1000) generan nulos.
    # Al final del dataset, el '.shift(-k)' de las etiquetas genera nulos porque mira al futuro.
    # Eliminamos estas filas extremas para que XGBoost no reciba datos incompletos.
    cleaned_df = full_df.drop_nulls()

    # 3. NORMALIZACIÓN OPTIONAL (Z-Score)
    # Nota técnica: Los modelos de árboles (XGBoost/LightGBM) NO necesitan normalización
    # porque cortan por umbrales independientes en cada variable.
    # Sin embargo, si en el futuro quieres usar estos mismos datos para una red neuronal
    # (como DeepLOB), activa 'apply_zscore=True'.
    if apply_zscore:
        feature_cols = [c for c in cleaned_df.columns if not c.startswith("label")]

        # Calculamos media y desviación estándar de cada variable y normalizamos
        z_score_exprs = [
            ((pl.col(col) - pl.col(col).mean()) / pl.col(col).std()).alias(col)
            for col in feature_cols
        ]
        # Mantener las columnas de etiquetas intactas sin normalizar
        label_cols = [pl.col(c) for c in cleaned_df.columns if c.startswith("label")]

        cleaned_df = cleaned_df.select(z_score_exprs + label_cols)

    return cleaned_df



In [ ]:
# Paso 4: Ensamblar y limpiar el dataset final
dataset_final = finalize_dataset_for_xgboost(X_features, y_labels, apply_zscore=False)


In [ ]:
dataset_final=dataset_filtrado.clone()

In [ ]:
dataset_final=dataset_final.drop([
    "label_k_10",
    "label_k_20",
    "label_k_30",
    "label_k_50"
])

In [62]:
dataset_final.collect().head(10)

v1,v2,v4,v6,v8,v10,v12,v14,v16,v18,v20,v22,v24,v26,v28,v30,v32,v34,v36,v38,v40,v41,v42,v43,v44,v45,v46,v51,v52,v53,v54,v55,v56,v61,v62,v63,v64,v65,v66,v71,v81,v91,v111,v112,v113,v114,v115,v116,v117,v118,v119,v120,v121,v122,v123,v124,v125,v126,v127,v128,v129,v130,v131,v133,v134,v135,v136,v138,v141,OBI_Nivel_1,label_k_100
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32
68227.1,3.26,6.072,0.002,0.001,0.004,0.002,0.137,0.002,0.001,0.002,0.004,0.006,0.002,0.003,0.001,0.011,0.004,0.002,0.001,0.009,0.1,0.3,1.2,2.0,2.8,3.1,0.0,0.1,0.7,1.3,1.4,1.5,0.0,0.1,0.4,0.6,1.3,1.5,0.0,0.0,0.0,0.0,0.017,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.007,0.0,0.0,0.036,0.1,87.754,0.1,0.1,0.301329,1
68227.1,3.209,6.07,0.002,0.001,0.004,0.002,0.137,0.002,0.001,0.002,0.004,0.006,0.002,0.003,0.001,0.011,0.004,0.002,0.001,0.009,0.1,0.3,1.2,2.0,2.8,3.1,0.0,0.1,0.7,1.3,1.4,1.5,0.0,0.1,0.4,0.6,1.3,1.5,0.0,0.0,0.0,-0.051,0.015,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.007,0.0,-0.051,-0.002,0.1,88.292,0.1,0.1,0.308331,1
68227.1,3.247,6.07,0.002,0.003,0.004,0.002,0.137,0.002,0.001,0.002,0.004,0.006,0.002,0.003,0.001,0.011,0.004,0.002,0.001,0.009,0.1,0.3,1.2,2.0,2.8,3.1,0.0,0.1,0.7,1.3,1.4,1.5,0.0,0.1,0.4,0.6,1.3,1.5,0.0,0.0,0.0,-0.013,0.015,0.0,0.002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.007,0.0,0.038,0.0,0.1,89.45,0.1,0.1,0.302995,1
68227.1,3.274,6.07,0.002,0.013,0.004,0.002,0.137,0.002,0.001,0.002,0.004,0.006,0.002,0.003,0.001,0.011,0.004,0.002,0.001,0.009,0.1,0.3,1.2,2.0,2.8,3.1,0.0,0.1,0.7,1.3,1.4,1.5,0.0,0.1,0.4,0.6,1.3,1.5,0.0,0.0,0.0,0.014,0.034,0.0,0.012,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.007,0.0,0.027,0.0,0.1,92.905,0.1,0.1,0.299229,1
68227.1,3.418,5.968,0.002,0.013,0.004,0.002,0.137,0.002,0.001,0.002,0.004,0.006,0.002,0.003,0.001,0.011,0.004,0.002,0.001,0.009,0.1,0.3,1.2,2.0,2.8,3.1,0.0,0.1,0.7,1.3,1.4,1.5,0.0,0.1,0.4,0.6,1.3,1.5,0.0,0.0,0.0,0.158,-0.068,0.0,0.012,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.007,0.0,0.144,-0.102,0.1,93.195,0.1,0.1,0.271681,1
68227.1,3.418,6.004,0.005,0.013,0.004,0.002,0.137,0.002,0.001,0.002,0.004,0.006,0.002,0.003,0.001,0.011,0.004,0.002,0.001,0.007,0.1,0.3,1.2,2.0,2.8,3.1,0.0,0.1,0.7,1.3,1.4,1.5,0.0,0.1,0.4,0.6,1.3,1.5,0.0,0.0,0.0,0.158,-0.068,0.003,0.012,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.002,0.0,0.0,0.036,0.1,93.302,0.1,0.1,0.274464,1
68227.1,3.418,5.465,0.005,0.013,0.004,0.002,0.137,0.002,0.001,0.002,0.004,0.006,0.002,0.003,0.001,0.011,0.004,0.002,0.001,0.007,0.1,0.3,1.2,2.0,2.8,3.1,0.0,0.1,0.7,1.3,1.4,1.5,0.0,0.1,0.4,0.6,1.3,1.5,0.0,0.0,0.0,0.209,-0.605,0.003,0.012,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.002,0.0,0.0,-0.539,0.1,92.87,0.1,0.1,0.23044,1
68220.1,0.002,0.028,0.002,0.446,0.002,0.649,0.894,2.228,0.003,0.895,0.047,3.78,0.002,0.126,0.002,0.002,0.002,0.002,0.002,0.003,0.1,2.5,4.4,5.6,5.8,6.2,0.0,1.1,2.2,2.6,2.7,3.0,0.0,1.3,2.1,2.9,3.0,3.1,0.0,0.0,-7.0,-3.245,-6.042,0.0,0.443,-0.002,0.647,0.757,2.226,0.002,0.893,0.043,3.774,0.0,0.123,0.001,-0.009,-0.002,0.0,0.001,-0.006,-7.0,-3.416,-5.437,0.1,83.585,0.1,0.1,0.866667,1
68220.1,0.812,3.505,0.002,0.003,0.002,0.042,0.002,0.012,0.002,0.001,0.001,0.002,0.002,0.001,0.001,1.998,0.141,0.167,0.003,0.125,0.1,0.3,2.1,2.5,3.3,4.2,0.0,0.1,0.9,1.1,1.5,1.7,0.0,0.1,1.1,1.3,1.7,2.4,0.0,0.0,-7.0,-2.462,-2.565,0.0,-0.01,-0.002,0.04,-0.135,0.01,0.001,-0.001,-0.003,-0.004,0.0,-0.002,0.0,1.987,0.137,0.165,0.002,0.116,0.0,0.81,3.477,0.1,78.606,0.1,0.1,0.623813,1


xgboost con validadcion

In [ ]:
import numpy as np
import polars as pl
import xgboost as xgb
import pickle
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# ==========================================
# 1. PREPARACIÓN DE MATRICES DESDE LAZYFRAME
# ==========================================
target_col = "label_k_100"

all_columns = dataset_final.collect_schema().names()
# Seleccionamos las variables a partir de v41 en adelante + OBI
feature_cols = [
    c for c in all_columns if c.startswith("v") ] + ["OBI_Nivel_1"]

# Traemos todo el DataFrame ordenado a memoria de forma eficiente
df_all = dataset_final.collect()
if "timestamp" in df_all.columns:
    df_all = df_all.sort("timestamp")

# Reservamos un 15% del total como un "Test Set Ciego" final (fuera de Hyperopt)
total_rows = len(df_all)
test_cutoff = int(total_rows * 0.85)

df_cv_pool = df_all.slice(0, test_cutoff)  # El pool que usará Cross-Validation
df_final_test = df_all.slice(test_cutoff)  # El test definitivo

# Extraemos las matrices del Test definitivo pasando los nombres de columnas
X_final_test = df_final_test.select(feature_cols).to_numpy()
y_final_test = df_final_test.select(target_col).to_numpy().squeeze().astype(int)
dtest_final = xgb.DMatrix(X_final_test, label=y_final_test, feature_names=feature_cols)

CONFIDENCE_THRESHOLD = 0.333334

# ==========================================
# 2. DEFINICIÓN DE FOLDS DE VALIDACIÓN TEMPORAL
# ==========================================
N_SPLITS = 4
cv_folds = []
chunk_size = len(df_cv_pool) // (N_SPLITS + 1)

for fold in range(N_SPLITS):
    train_end_idx = chunk_size * (fold + 1)
    val_end_idx = train_end_idx + chunk_size

    train_fold = df_cv_pool.slice(0, train_end_idx)
    val_fold = df_cv_pool.slice(train_end_idx, chunk_size)

    X_tr = train_fold.select(feature_cols).to_numpy()
    y_tr = train_fold.select(target_col).to_numpy().squeeze().astype(int)

    X_va = val_fold.select(feature_cols).to_numpy()
    y_va = val_fold.select(target_col).to_numpy().squeeze().astype(int)

    clases, counts = np.unique(y_tr, return_counts=True)
    class_weights = len(y_tr) / (3.0 * counts)
    sw_tr = np.array([class_weights[y] for y in y_tr])

    # Se asocian las columnas para evitar errores de alineación en GPU
    dm_train = xgb.DMatrix(X_tr, label=y_tr, weight=sw_tr, feature_names=feature_cols)
    dm_val = xgb.DMatrix(X_va, label=y_va, feature_names=feature_cols)

    cv_folds.append((dm_train, dm_val, y_va))

# ==========================================
# 3. ESPACIO DE HIPERPARÁMETROS (CORREGIDO)
# ==========================================
space = {
    # Cambiado hp.choice por hp.quniform para evitar el error del índice '0'
    "max_depth": hp.quniform("max_depth", 3, 9, 1),
    "learning_rate": hp.uniform("learning_rate", 0.01, 0.12),
    "num_boost_round": hp.quniform("num_boost_round", 100, 300, 50),
    "subsample": hp.uniform("subsample", 0.6, 0.85),
    "colsample_bytree": hp.uniform("colsample_bytree", 0.6, 0.85),
    "max_delta_step": hp.quniform("max_delta_step", 1, 7, 1),

    # Límites suavizados para asegurar que XGBoost SÍ construya árboles reales
    "min_child_weight": hp.uniform("min_child_weight", 1, 15),  # Bajado de (10, 100)
    "reg_alpha": hp.uniform("reg_alpha", 0.0, 5.0),             # Bajado de (0.0, 15.0)
    "reg_lambda": hp.uniform("reg_lambda", 1.0, 5.0),           # Bajado de (1.0, 20.0)
    "gamma": hp.uniform("gamma", 0.0, 1.0),                     # Bajado de (0.5, 5.0)
}

# ==========================================
# 4. FUNCIÓN OBJETIVO CON VALIDACIÓN CRUZADA
# ==========================================
def objective(space_params):
    fixed_params = {
        "objective": "multi:softprob",
        "num_class": 3,
        "tree_method": "hist",
        "device": "cuda",
    }
    current_params = {**fixed_params, **space_params}

    # Conversión directa y segura a enteros
    current_params["max_depth"] = int(current_params["max_depth"])
    current_params["max_delta_step"] = int(current_params["max_delta_step"])
    n_rounds = int(current_params.pop("num_boost_round"))

    f1_scores_folds = []

    for dm_train, dm_val, y_val_true in cv_folds:
        model = xgb.train(current_params, dm_train, num_boost_round=n_rounds)
        preds_prob = model.predict(dm_val)

        y_pred_tmp = np.ones(len(preds_prob), dtype=int)
        for i in range(len(preds_prob)):
            if preds_prob[i][0] >= CONFIDENCE_THRESHOLD:
                y_pred_tmp[i] = 0
            elif preds_prob[i][2] >= CONFIDENCE_THRESHOLD:
                y_pred_tmp[i] = 2

        f1_fold = f1_score(y_val_true, y_pred_tmp, labels=[0, 2], average="macro", zero_division=0)
        f1_scores_folds.append(f1_fold)

    mean_f1_cv = np.mean(f1_scores_folds)
    return {"loss": -mean_f1_cv, "status": STATUS_OK}

# ==========================================
# 5. EJECUCIÓN DE LA OPTIMIZACIÓN (CORREGIDA)
# ==========================================
print(f"🚀 Iniciando Optimización Bayesiana con Validación Cruzada Temporal ({N_SPLITS} Folds)...")
trials = Trials()
best_hyperparams = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=40, trials=trials)

# Extracción directa de parámetros de Hyperopt convirtiendo a enteros limpios
final_rounds = int(best_hyperparams.pop("num_boost_round"))
best_hyperparams["max_depth"] = int(best_hyperparams["max_depth"])
best_hyperparams["max_delta_step"] = int(best_hyperparams["max_delta_step"])

# ==========================================
# 6. EVALUACIÓN DE ROBUSTEZ EN EL TEST CIEGO
# ==========================================
print("\n El entrenamiento de Hyperopt ha terminado. Reentrenando modelo final...")

X_cv_pool = df_cv_pool.select(feature_cols).to_numpy()
y_cv_pool = df_cv_pool.select(target_col).to_numpy().squeeze().astype(int)

clases, counts = np.unique(y_cv_pool, return_counts=True)
class_weights = len(y_cv_pool) / (3.0 * counts)
sw_cv_pool = np.array([class_weights[y] for y in y_cv_pool])

# FIJADO: Vinculamos explícitamente feature_names aquí
dtrain_full = xgb.DMatrix(X_cv_pool, label=y_cv_pool, weight=sw_cv_pool, feature_names=feature_cols)

final_params = {
    "objective": "multi:softprob",
    "num_class": 3,
    "tree_method": "hist",
    "device": "cuda",
    **best_hyperparams,
}

# Entrenar el modelo definitivo real
final_model = xgb.train(final_params, dtrain_full, num_boost_round=final_rounds)

# ==========================================
# 7. GUARDADO COMPLEMENTARIO E INDESTRUCTIBLE
# ==========================================
nombre_modelo_pkl = "final_xgb_model.pkl"
with open(nombre_modelo_pkl, "wb") as f:
    pickle.dump(final_model, f)

print(f"✅ ¡Modelo guardado con éxito en '{nombre_modelo_pkl}' usando Pickle!")

# Validar cantidad de árboles creados para auditar que ya NO está vacío
num_arboles = len(final_model.get_dump())
print(f"🔍 Auditoría técnica: El modelo contiene {num_arboles} árboles de decisión reales.")

# ==========================================
# 8. REPORTE DE MÉTRICAS VERDADERAS
# ==========================================
# Forzar recálculo limpio sobre dtest_final
final_preds_prob = final_model.predict(dtest_final)

y_pred_final = np.ones(len(final_preds_prob), dtype=int)
for i in range(len(final_preds_prob)):
    if final_preds_prob[i][0] >= CONFIDENCE_THRESHOLD: y_pred_final[i] = 0
    elif final_preds_prob[i][2] >= CONFIDENCE_THRESHOLD: y_pred_final[i] = 2

print("\n📊 REPORTE DE CLASIFICACIÓN FINAL REAL (Sobre Datos Ciegos):")
print(classification_report(y_final_test, y_pred_final, target_names=["DOWN", "STAT", "UP"], zero_division=0))

print("\n🧮 MATRIZ DE CONFUSIÓN REAL:")
print(confusion_matrix(y_final_test, y_pred_final))


🚀 Iniciando Optimización Bayesiana con Validación Cruzada Temporal (4 Folds)...
100%|██████████| 40/40 [24:55<00:00, 37.38s/trial, best loss: -0.3857875317039244]

 El entrenamiento de Hyperopt ha terminado. Reentrenando modelo final...
✅ ¡Modelo guardado con éxito en 'final_xgb_model.pkl' usando Pickle!
🔍 Auditoría técnica: El modelo contiene 900 árboles de decisión reales.

📊 REPORTE DE CLASIFICACIÓN FINAL REAL (Sobre Datos Ciegos):
              precision    recall  f1-score   support

        DOWN       0.55      0.26      0.35     58480
        STAT       0.47      0.79      0.59     89576
          UP       0.57      0.29      0.39     61011

    accuracy                           0.49    209067
   macro avg       0.53      0.45      0.44    209067
weighted avg       0.52      0.49      0.46    209067


🧮 MATRIZ DE CONFUSIÓN REAL:
[[14967 40018  3495]
 [ 9253 70527  9796]
 [ 3051 40049 17911]]


In [ ]:
# ==========================================
# 9. EVALUACIÓN DETALLADA DE DIFERENTES UMBRALES
# ==========================================
print("\n🔄 Generando reportes completos para diferentes umbrales de confianza...")

# Definir los umbrales desde 0.45 hasta 0.80 de 5 en 5
umbrales = np.arange(0.45, 0.81, 0.05)

for u in umbrales:
    print("\n" + "="*80)
    print(f"🎯 EVALUACIÓN CON UMBRAL DE CONFIANZA: {u:.2f}")
    print("="*80)

    # Inicializar todas las predicciones como STAT (1)
    y_pred_u = np.ones(len(final_preds_prob), dtype=int)

    # Aplicar el umbral actual
    for i in range(len(final_preds_prob)):
        if final_preds_prob[i][0] >= u:
            y_pred_u[i] = 0  # DOWN
        elif final_preds_prob[i][2] >= u:
            y_pred_u[i] = 2  # UP

    # Calcular el porcentaje de ticks donde el modelo decidió operar (DOWN o UP)
    ticks_operados = np.sum((y_pred_u == 0) | (y_pred_u == 2))
    pct_actividad = 100 * ticks_operados / len(y_pred_u)

    print(f"\n📊 REPORTE DE CLASIFICACIÓN FINAL REAL (Sobre Datos Ciegos - Umbral {u:.2f}):")
    print(classification_report(y_final_test, y_pred_u, target_names=["DOWN", "STAT", "UP"], zero_division=0))

    print(f"🧮 MATRIZ DE CONFUSIÓN REAL (Umbral {u:.2f}):")
    print(confusion_matrix(y_final_test, y_pred_u))

    print(f"\n📈 Grado de actividad en mercado: {pct_actividad:.2f}% de los ticks operados.")



🔄 Generando reportes completos para diferentes umbrales de confianza...

🎯 EVALUACIÓN CON UMBRAL DE CONFIANZA: 0.45

📊 REPORTE DE CLASIFICACIÓN FINAL REAL (Sobre Datos Ciegos - Umbral 0.45):
              precision    recall  f1-score   support

        DOWN       0.49      0.40      0.44     58480
        STAT       0.48      0.57      0.52     89576
          UP       0.50      0.46      0.48     61011

    accuracy                           0.49    209067
   macro avg       0.49      0.48      0.48    209067
weighted avg       0.49      0.49      0.49    209067

🧮 MATRIZ DE CONFUSIÓN REAL (Umbral 0.45):
[[23669 27464  7347]
 [18217 50972 20387]
 [ 6308 26718 27985]]

📈 Grado de actividad en mercado: 49.70% de los ticks operados.

🎯 EVALUACIÓN CON UMBRAL DE CONFIANZA: 0.50

📊 REPORTE DE CLASIFICACIÓN FINAL REAL (Sobre Datos Ciegos - Umbral 0.50):
              precision    recall  f1-score   support

        DOWN       0.52      0.33      0.41     58480
        STAT       0.48      

lightboost con val

In [ ]:
import lightgbm as lgb
import numpy as np
import polars as pl
import pickle
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# ==========================================
# 1. PREPARACIÓN DE MATRICES DESDE LAZYFRAME
# ==========================================
target_col = "label_k_100"

all_columns = dataset_final.collect_schema().names()
feature_cols = [
    c for c in all_columns if c.startswith("v")
] + ["OBI_Nivel_1"]

# Cargamos todo el DataFrame en memoria de manera ordenada
df_all = dataset_final.collect()
if "timestamp" in df_all.columns:
    df_all = df_all.sort("timestamp")

# Aislamos el 15% final como Test Ciego Absoluto (Out-of-Sample)
total_rows = len(df_all)
test_cutoff = int(total_rows * 0.85)

df_cv_pool = df_all.slice(0, test_cutoff)  # Pool para la validación cruzada temporal
df_final_test = df_all.slice(test_cutoff)  # El test definitivo intocado por Hyperopt

X_final_test = df_final_test.select(feature_cols).to_numpy()
y_final_test = df_final_test.select(target_col).to_numpy().squeeze().astype(int)

OPTIMIZATION_THRESHOLD = 0.33333334

# ==========================================
# 2. CREACIÓN DE LOS FOLDS DE VALIDACIÓN TEMPORAL
# ==========================================
N_SPLITS = 4
cv_folds = []
chunk_size = len(df_cv_pool) // (N_SPLITS + 1)

print(f"📦 Generando {N_SPLITS} Folds expansivos para series temporales...")

for fold in range(N_SPLITS):
    train_end_idx = chunk_size * (fold + 1)

    train_fold = df_cv_pool.slice(0, train_end_idx)
    val_fold = df_cv_pool.slice(train_end_idx, chunk_size)

    X_tr = train_fold.select(feature_cols).to_numpy()
    y_tr = train_fold.select(target_col).to_numpy().squeeze().astype(int)

    X_va = val_fold.select(feature_cols).to_numpy()
    y_va = val_fold.select(target_col).to_numpy().squeeze().astype(int)

    # Pesos balanceados inversos específicos de este bloque temporal
    clases, counts = np.unique(y_tr, return_counts=True)
    class_weights = len(y_tr) / (3.0 * counts)
    sw_tr = np.array([class_weights[y] for y in y_tr])

    # Construcción de los objetos Dataset nativos de LightGBM
    train_dataset = lgb.Dataset(X_tr, label=y_tr, weight=sw_tr)

    cv_folds.append((train_dataset, X_va, y_va))

# ==========================================
# 3. CONFIGURACIÓN DEL ESPACIO DE HYPEROPT ACELERADO
# ==========================================
space_lgb = {
    "num_leaves": hp.quniform("num_leaves", 20, 60, 5),
    "max_depth": hp.quniform("max_depth", 4, 8, 1),
    "learning_rate": hp.uniform("learning_rate", 0.02, 0.12),
    "feature_fraction": hp.uniform("feature_fraction", 0.6, 0.9),
    "bagging_fraction": hp.uniform("bagging_fraction", 0.6, 0.9),
    "min_child_samples": hp.quniform("min_child_samples", 50, 300, 50),
    "reg_alpha": hp.uniform("reg_alpha", 0.0, 5.0),
    "reg_lambda": hp.uniform("reg_lambda", 0.0, 5.0),
}

# ==========================================
# 4. FUNCIÓN OBJETIVO ACELERADA EN GPU
# ==========================================
def objective_lgb(space_params):
    params = {
        "objective": "multiclass",
        "num_class": 3,
        "metric": "multi_logloss",
        "boosting_type": "gbdt",
        "device": "gpu",
        "gpu_use_dp": False,
        "verbosity": -1,
        "feature_pre_filter": False,
        "bagging_freq": 1,
        **space_params
    }
    params["num_leaves"] = int(params["num_leaves"])
    params["max_depth"] = int(params["max_depth"])
    params["min_child_samples"] = int(params["min_child_samples"])

    f1_scores_folds = []

    for train_data_fold, X_va, y_va_true in cv_folds:
        model = lgb.train(params, train_data_fold, num_boost_round=80)
        preds_prob = model.predict(X_va)

        y_pred_tmp = np.ones(len(preds_prob), dtype=int)
        y_pred_tmp[preds_prob[:, 0] >= OPTIMIZATION_THRESHOLD] = 0
        y_pred_tmp[preds_prob[:, 2] >= OPTIMIZATION_THRESHOLD] = 2

        # FIJADO: Añadido [0, 2] en el parámetro labels
        f1_fold = f1_score(y_va_true, y_pred_tmp, labels=[0, 2], average="macro", zero_division=0)
        f1_scores_folds.append(f1_fold)

    mean_f1_cv = np.mean(f1_scores_folds)
    return {"loss": -mean_f1_cv, "status": STATUS_OK}

# ==========================================
# 5. EJECUCIÓN DE LA OPTIMIZACIÓN
# ==========================================
print("🚀 Iniciando optimización bayesiana acelerada para LightGBM (GPU)...")
trials_lgb = Trials()
best_lgb = fmin(fn=objective_lgb, space=space_lgb, algo=tpe.suggest, max_evals=40, trials=trials_lgb)

# Conversión limpia de tipos
best_lgb["num_leaves"] = int(best_lgb["num_leaves"])
best_lgb["max_depth"] = int(best_lgb["max_depth"])
best_lgb["min_child_samples"] = int(best_lgb["min_child_samples"])

print("\n🏆 Configuración óptima encontrada para LightGBM:")
print(best_lgb)

# ==========================================
# 6. ENTRENAMIENTO COMPLETO EN EL POOL DE VALIDACIÓN
# ==========================================
print("\n🧠 Entrenando LightGBM final sobre el pool completo...")
X_cv_pool = df_cv_pool.select(feature_cols).to_numpy()
y_cv_pool = df_cv_pool.select(target_col).to_numpy().squeeze().astype(int)

clases, counts = np.unique(y_cv_pool, return_counts=True)
class_weights = len(y_cv_pool) / (3.0 * counts)
sw_cv_pool = np.array([class_weights[y] for y in y_cv_pool])

train_data_full = lgb.Dataset(X_cv_pool, label=y_cv_pool, weight=sw_cv_pool)

final_params = {
    "objective": "multiclass",
    "num_class": 3,
    "metric": "multi_logloss",
    "boosting_type": "gbdt",
    "device": "gpu",
    "gpu_use_dp": False,
    "n_jobs": -1,
    "verbosity": -1,
    "feature_pre_filter": False,
    "bagging_freq": 1,
    **best_lgb
}

final_lgb_model = lgb.train(final_params, train_data_full, num_boost_round=150)

# Guardar el modelo en un archivo binario Pickle inmediatamente para no perderlo
with open("final_lgb_model.pkl", "wb") as f:
    pickle.dump(final_lgb_model, f)
print("✅ ¡Modelo LightGBM guardado con éxito como 'final_lgb_model.pkl'!")

preds_prob_lgb = final_lgb_model.predict(X_final_test)

# ==========================================
# 7. INFERENCIA Y EVALUACIÓN SEPARADA MULTI-UMBRAL
# ==========================================
print("\n🔄 Generando reportes completos para diferentes umbrales de confianza (45% a 80%)...")

umbrales = np.arange(0.45, 0.81, 0.05)

prob_down = preds_prob_lgb[:, 0]
prob_up = preds_prob_lgb[:, 2]

for u in umbrales:
    print("\n" + "="*80)
    print(f"🎯 EVALUACIÓN CON UMBRAL DE CONFIANZA: {u:.2f}")
    print("="*80)

    y_pred_u = np.ones(len(preds_prob_lgb), dtype=int)

    y_pred_u[prob_down >= u] = 0
    y_pred_u[prob_up >= u] = 2

    print(f"\n📊 REPORTE DE CLASIFICACIÓN FINAL REAL (Sobre Datos Ciegos - Umbral {u:.2f}):")
    print(classification_report(y_final_test, y_pred_u, target_names=["DOWN", "STAT", "UP"], zero_division=0))

    print(f"🧮 MATRIZ DE CONFUSIÓN REAL (Umbral {u:.2f}):")
    print(confusion_matrix(y_final_test, y_pred_u))

    ticks_operados = np.sum((y_pred_u == 0) | (y_pred_u == 2))
    pct_operado = (ticks_operados / len(y_final_test)) * 100
    print(f"\n📈 Grado de actividad en mercado: {pct_operado:.2f}% de los ticks operados.")


📦 Generando 4 Folds expansivos para series temporales...
🚀 Iniciando optimización bayesiana acelerada para LightGBM (GPU)...
100%|██████████| 40/40 [54:01<00:00, 81.04s/trial, best loss: -0.3578639947487022]

🏆 Configuración óptima encontrada para LightGBM:
{'bagging_fraction': np.float64(0.7584754515124938), 'feature_fraction': np.float64(0.8953132502811604), 'learning_rate': np.float64(0.11421273375098262), 'max_depth': 8, 'min_child_samples': 300, 'num_leaves': 50, 'reg_alpha': np.float64(1.079476233940872), 'reg_lambda': np.float64(2.284937733927958)}

🧠 Entrenando LightGBM final sobre el pool completo...
✅ ¡Modelo LightGBM guardado con éxito como 'final_lgb_model.pkl'!

🔄 Generando reportes completos para diferentes umbrales de confianza (45% a 80%)...

🎯 EVALUACIÓN CON UMBRAL DE CONFIANZA: 0.45

📊 REPORTE DE CLASIFICACIÓN FINAL REAL (Sobre Datos Ciegos - Umbral 0.45):
              precision    recall  f1-score   support

        DOWN       0.52      0.39      0.44     58480
    

In [ ]:
# ==========================================
# 7. INFERENCIA Y EVALUACIÓN SEPARADA MULTI-UMBRAL
# ==========================================
print("\n🔄 Generando reportes completos para diferentes umbrales de confianza (45% a 80%)...")

# Definir los umbrales desde 0.45 hasta 0.80 de 5 en 5
umbrales = np.arange(0.45, 0.81, 0.05)

prob_down = preds_prob_lgb[:, 0]
prob_up = preds_prob_lgb[:, 2]

for u in umbrales:
    print("\n" + "="*80)
    print(f"🎯 EVALUACIÓN CON UMBRAL DE CONFIANZA: {u:.2f}")
    print("="*80)

    # Inicializar todas las predicciones como STAT (1)
    y_pred_u = np.ones(len(preds_prob_lgb), dtype=int)

    # Aplicar el umbral actual
    y_pred_u[prob_down >= u] = 0  # DOWN
    y_pred_u[prob_up >= u] = 2    # UP

    print(f"\n📊 REPORTE DE CLASIFICACIÓN FINAL REAL (Sobre Datos Ciegos - Umbral {u:.2f}):")
    print(classification_report(y_final_test, y_pred_u, target_names=["DOWN", "STAT", "UP"], zero_division=0))

    print(f"🧮 MATRIZ DE CONFUSIÓN REAL (Umbral {u:.2f}):")
    print(confusion_matrix(y_final_test, y_pred_u))

    # Calcular el grado de actividad operativa en el LOB
    ticks_operados = np.sum((y_pred_u == 0) | (y_pred_u == 2))
    pct_operado = (ticks_operados / len(y_final_test)) * 100
    print(f"\n📈 Grado de actividad en mercado: {pct_operado:.2f}% de los ticks operados.")



🔄 Generando reportes completos para diferentes umbrales de confianza (45% a 80%)...

🎯 EVALUACIÓN CON UMBRAL DE CONFIANZA: 0.45

📊 REPORTE DE CLASIFICACIÓN FINAL REAL (Sobre Datos Ciegos - Umbral 0.45):
              precision    recall  f1-score   support

        DOWN       0.52      0.39      0.44     58480
        STAT       0.49      0.63      0.55     89576
          UP       0.52      0.43      0.47     61011

    accuracy                           0.50    209067
   macro avg       0.51      0.48      0.49    209067
weighted avg       0.51      0.50      0.50    209067

🧮 MATRIZ DE CONFUSIÓN REAL (Umbral 0.45):
[[22636 29535  6309]
 [15353 56762 17461]
 [ 5507 29415 26089]]

📈 Grado de actividad en mercado: 44.65% de los ticks operados.

🎯 EVALUACIÓN CON UMBRAL DE CONFIANZA: 0.50

📊 REPORTE DE CLASIFICACIÓN FINAL REAL (Sobre Datos Ciegos - Umbral 0.50):
              precision    recall  f1-score   support

        DOWN       0.56      0.31      0.40     58480
        STAT     

In [ ]:
import joblib

# 1. Guardar el modelo en su formato nativo de texto
final_lgb_model.save_model('final_model_lgb_real.ubj')
print("💾 Modelo LightGBM 'final_model' guardado correctamente.")

💾 Modelo LightGBM 'final_model' guardado correctamente.
